# 03 — Model Evaluation & Figures
**Alzheimer's Disease Classification | KLE Technological University**

Loads saved models and generates all evaluation figures:
- Confusion Matrix (multiclass)
- ROC Curve (AD one-vs-rest)
- Precision-Recall Curve
- Feature Importance (weighted ensemble)
- Probability Density by Class
- Threshold vs Accuracy / F1

### Input
- Saved model artifacts from `02_model_training.ipynb`
- `ADNI_MRI_Cognitive_Merged.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, roc_curve, auc,
    precision_recall_curve, accuracy_score,
    classification_report, f1_score, precision_score, recall_score
)
import warnings
warnings.filterwarnings('ignore')

MERGED_PATH = '/content/drive/MyDrive/AD1/ADNI_MRI_Cognitive_Merged.csv'
MODELS_DIR  = '/content/drive/MyDrive/AD1/lightgbm_final'
FIGURES_DIR = '/content/drive/MyDrive/AD1/lightgbm_final/eval_figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# Load models and params
model_cog  = joblib.load(os.path.join(MODELS_DIR, 'lgb_model_cog.joblib'))
model_mri  = joblib.load(os.path.join(MODELS_DIR, 'lgb_model_mri.joblib'))
scaler_cog = joblib.load(os.path.join(MODELS_DIR, 'scaler_cog.joblib'))
scaler_mri = joblib.load(os.path.join(MODELS_DIR, 'scaler_mri.joblib'))
ens_params = joblib.load(os.path.join(MODELS_DIR, 'ensemble_params.joblib'))

w_cog             = ens_params['w_cog']
w_mri             = ens_params['w_mri']
chosen_threshold  = ens_params['chosen_threshold']
print('Models loaded. Threshold:', chosen_threshold)

In [ ]:
# Rebuild test set (same seed as training)
df = pd.read_csv(MERGED_PATH, low_memory=False)
if 'APOE4' in df.columns:
    df = df.drop(columns=['APOE4'])

label_map = {'CN':0,'MCI':1,'LMCI':1,'EMCI':1,'SMC':0,'AD':2}
df['DX_num'] = df['DX_bl'].map(label_map)
df = df.dropna(subset=['DX_num'])
df['DX_num'] = df['DX_num'].astype(int)

df['ADAS_gap']  = df['MMSE'] - df['ADAS13']
df['AGE_sq']    = df['AGE'] ** 2
df['H_E_ratio'] = df['Hippocampus'] / (df['Entorhinal'] + 1e-9)

cog_features = ['AGE','AGE_sq','PTEDUCAT','MMSE','ADAS13','CDRSB','ADAS_gap']
mri_features = ['Hippocampus','Entorhinal','H_E_ratio']

mask = df[cog_features + mri_features].notna().all(axis=1)
df   = df[mask].reset_index(drop=True)

Xc = df[cog_features]; Xm = df[mri_features]; y = df['DX_num']
_, Xc_test, _, Xm_test, _, y_test = train_test_split(
    Xc, Xm, y, test_size=0.20, random_state=42, stratify=y
)
Xc_test_s = pd.DataFrame(scaler_cog.transform(Xc_test), columns=cog_features, index=Xc_test.index)
Xm_test_s = pd.DataFrame(scaler_mri.transform(Xm_test), columns=mri_features, index=Xm_test.index)

probs_cog = model_cog.predict_proba(Xc_test_s)
probs_mri = model_mri.predict_proba(Xm_test_s)
probs_ens = w_cog * probs_cog + w_mri * probs_mri
y_pred    = np.argmax(probs_ens, axis=1)
print('Test set rebuilt. Samples:', len(y_test))

In [ ]:
# --- Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['CN','MCI','AD'], yticklabels=['CN','MCI','AD'])
plt.xlabel('Predicted'); plt.ylabel('True')
plt.title('Confusion Matrix (Multiclass)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'confusion_matrix_multiclass.png'), dpi=150)
plt.show()

In [ ]:
# --- ROC Curve (AD one-vs-rest) ---
y_test_ad = (y_test == 2).astype(int)
ad_prob   = probs_ens[:, 2]
fpr, tpr, _ = roc_curve(y_test_ad, ad_prob)
roc_auc_val  = auc(fpr, tpr)

plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, lw=2, label=f'AD ROC (AUC = {roc_auc_val:.3f})')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — AD (one-vs-rest)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'roc_ad.png'), dpi=150)
plt.show()
print(f'AUC: {roc_auc_val:.4f}')

In [ ]:
# --- Precision-Recall Curve ---
prec, rec, thrs = precision_recall_curve(y_test_ad, ad_prob)

plt.figure(figsize=(7,5))
plt.plot(rec, prec, lw=2, color='darkorange')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title('Precision-Recall Curve — AD')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'pr_ad.png'), dpi=150)
plt.show()

In [ ]:
# --- Weighted Feature Importance ---
imp_cog = pd.Series(model_cog.feature_importances_, index=cog_features)
imp_mri = pd.Series(model_mri.feature_importances_, index=mri_features)
imp_all = pd.concat([imp_cog * w_cog, imp_mri * w_mri]).sort_values()

plt.figure(figsize=(9,6))
colors = plt.cm.viridis_r(np.linspace(0.1, 0.9, len(imp_all)))
imp_all.plot(kind='barh', color=colors)
plt.xlabel('Weighted importance (arb)')
plt.title('Weighted Ensemble Feature Importances')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'feature_importance_weighted.png'), dpi=150)
plt.show()

In [ ]:
# --- Threshold vs Accuracy / Precision / Recall / F1 ---
thrs_range = np.arange(0.05, 0.95, 0.01)
accs, precs, recs, f1s = [], [], [], []

for t in thrs_range:
    y_adj = np.where(ad_prob >= t, 2, np.argmax(probs_ens[:, :2], axis=1))
    accs.append(accuracy_score(y_test, y_adj))
    y_bin = (y_adj == 2).astype(int)
    precs.append(precision_score(y_test_ad, y_bin, zero_division=0))
    recs.append(recall_score(y_test_ad, y_bin, zero_division=0))
    f1s.append(f1_score(y_test_ad, y_bin, zero_division=0))

plt.figure(figsize=(9,5))
plt.plot(thrs_range, precs, label='Precision')
plt.plot(thrs_range, recs,  label='Recall')
plt.plot(thrs_range, f1s,   label='F1-score')
plt.axvline(x=0.50, color='red', linestyle='--', label='Chosen threshold = 0.5')
plt.xlabel('AD Probability Threshold'); plt.ylabel('Metric value')
plt.title('Precision, Recall and F1 vs Threshold')
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'threshold_vs_metrics.png'), dpi=150)
plt.show()

In [ ]:
# --- Final metrics summary ---
print('=== Final Model Metrics ===')
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('AUC (OvR):', round(roc_auc_val, 4))
print('\nFull Classification Report:')
print(classification_report(y_test, y_pred, target_names=['CN','MCI','AD']))
print(f'\nAll figures saved to: {FIGURES_DIR}')